In [ ]:
# Colab dependency install (T4 / CUDA + QLoRA)
!pip install -U bitsandbytes transformers peft datasets evaluate bert-score rouge_score huggingface_hub trl accelerate tensorboard

In [ ]:
# Imports
import math
import os
import random
import re
from dataclasses import dataclass
from functools import partial
from typing import Any

import evaluate
import numpy as np
import torch
from datasets import concatenate_datasets, load_dataset
from google.colab import userdata
# from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    TrainerCallback,
)
from trl import SFTConfig, SFTTrainer

In [ ]:
# Authentication + runtime checks
hf_token = userdata.get("HF_TOKEN_WRITE") # For Google Colab
# hf_token = UserSecretsClient().get_secret("HF_TOKEN") # For Kaggle Notebook
if hf_token:
    login(token=hf_token)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA runtime not detected. Set Colab runtime to GPU.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
IS_H100 = "h100" in GPU_NAME.lower() or GPU_CAPABILITY[0] >= 9

print(f"CUDA device: {GPU_NAME} (compute capability: {GPU_CAPABILITY[0]}.{GPU_CAPABILITY[1]})")
print(f"H100 profile enabled: {IS_H100}")

# Throughput-oriented CUDA settings for Hopper/Ampere.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

CUDA device: NVIDIA L4 (compute capability: 8.9)
H100 profile enabled: False


In [ ]:
# Configuration
# -----------------------------------------------------------------------------
# Core experiment settings
# -----------------------------------------------------------------------------
SYSTEM_PROMPT_TEMPLATE = """
You are an ambient clinical documentation assistant. Given a verbatim
transcript of a doctor-patient encounter, generate a structured
SOAP-style outpatient progress note.

### Guidelines
- Use clear ALL-CAPS section headings appropriate to the content
  (e.g., CHIEF COMPLAINT, HISTORY OF PRESENT ILLNESS, MEDICAL HISTORY,
  REVIEW OF SYSTEMS, VITALS, PHYSICAL EXAM, RESULTS, ASSESSMENT AND PLAN,
  INSTRUCTIONS).
- Write in a professional clinical style using third-person narrative prose.
- Include only sections for which information is present in the dialogue.
- Do not fabricate any information not explicitly stated in the dialogue.
- Preserve all medication names, doses, and frequencies exactly as spoken.
- The input transcript contains no speaker labels — infer clinical content
  from context.
""".strip()


@dataclass(frozen=True)
class ExperimentConfig:
    """Centralized runtime/training configuration for notebook readability."""

    seed: int
    model_id: str
    system_prompt: str
    train_noise_prob: float
    valid_noise_prob: float
    test_noise_prob: float
    max_seq_length: int
    max_new_tokens: int
    gen_repetition_penalty: float
    gen_no_repeat_ngram_size: int
    num_train_epochs: int
    learning_rate: float
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    gradient_accumulation_steps: int
    warmup_steps: int
    max_grad_norm: float
    eval_steps: int
    save_steps: int
    logging_steps: int
    early_stopping_patience: int
    eval_subset_size: int
    test_subset_size: int
    output_dir: str
    push_to_hub: bool
    hub_model_id: str
    report_to: str
    num_dataloader_workers: int
    verbose_eval: bool
    verbose_eval_samples: int

    @classmethod
    def from_runtime(cls, is_h100: bool) -> "ExperimentConfig":
        """Build config values that depend on GPU/runtime profile."""
        return cls(
            seed=42,
            model_id="google/medgemma-1.5-4b-it",
            system_prompt=SYSTEM_PROMPT_TEMPLATE,
            train_noise_prob=0.02,
            valid_noise_prob=0.0,
            test_noise_prob=0.0,
            max_seq_length=4096,
            max_new_tokens=1024,
            gen_repetition_penalty=1.15,
            gen_no_repeat_ngram_size=4,
            num_train_epochs=12,
            learning_rate=1e-4,
            per_device_train_batch_size=4 if is_h100 else 1,
            per_device_eval_batch_size=4 if is_h100 else 1,
            gradient_accumulation_steps=1 if is_h100 else 4,
            warmup_steps=10,
            max_grad_norm=0.3,
            eval_steps=5,
            save_steps=5,
            logging_steps=5,
            early_stopping_patience=3,
            eval_subset_size=128,
            test_subset_size=256,
            output_dir="outputs/medgemma-aci-lora",
            push_to_hub=True,
            hub_model_id="gabrielbuzzi/medgemma-aci-lora",
            report_to="tensorboard",
            num_dataloader_workers=max(4, min(16, (os.cpu_count() or 8) // 2)) if is_h100 else 2,
            verbose_eval=True,
            verbose_eval_samples=3,
        )


CFG = ExperimentConfig.from_runtime(IS_H100)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

In [ ]:
# Dataset load + split selection
dataset = load_dataset("mkieffer/ACI-Bench", "aci")
train_ds = dataset["train"]
valid_ds = dataset["valid"]

print(train_ds)
print(valid_ds)

README.md: 0.00B [00:00, ?B/s]

aci/train.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

aci/valid.parquet:   0%|          | 0.00/59.8k [00:00<?, ?B/s]

aci/test1.parquet:   0%|          | 0.00/98.3k [00:00<?, ?B/s]

aci/test2.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

...

Generating valid split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating test1 split:   0%|          | 0/22 [00:00<?, ? examples/s]

Generating test2 split:   0%|          | 0/22 [00:00<?, ? examples/s]

Generating test3 split:   0%|          | 0/22 [00:00<?, ? examples/s]

Dataset({
    features: ['encounter_id', 'dialogue', 'note'],
    num_rows: 35
})
Dataset({
    features: ['encounter_id', 'dialogue', 'note'],
    num_rows: 11
})


In [ ]:
# Preprocessing helpers

def clean_dialogue(text: str) -> str:
    """Normalize transcript text to non-diarized single-line format."""
    text = re.sub(r"\[doctor\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[patient\]", "", text, flags=re.IGNORECASE)
    text = text.replace("\\n", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def simulate_transcription_noise(text: str, drop_prob: float) -> str:
    """Drop random words to approximate ASR omissions."""
    words = text.split()
    words = [w for w in words if random.random() > drop_prob]
    return " ".join(words)


def build_dialogue_user_content(dialogue: str) -> str:
    """Build user payload required by prompt contract."""
    return f"<dialogue>\n{dialogue}\n</dialogue>"


def wrap_note(note: str) -> str:
    """Wrap reference target note with XML-style tags."""
    return f"<note>\n{note.strip()}\n</note>"


def build_messages(dialogue: str, note: str | None = None) -> list[dict[str, str]]:
    """Create chat messages using a single canonical structure for all stages."""
    messages = [
        {"role": "system", "content": CFG.system_prompt},
        {"role": "user", "content": build_dialogue_user_content(dialogue)},
    ]
    if note is not None:
        messages.append({"role": "assistant", "content": wrap_note(note)})
    return messages


def extract_prompt_messages(messages: list[dict[str, str]]) -> list[dict[str, str]]:
    """Return only prompt-side messages (system + user)."""
    return [msg for msg in messages if msg["role"] in {"system", "user"}]


def build_prompt_text(tokenizer_obj: AutoTokenizer, messages: list[dict[str, str]]) -> str:
    """Render messages with tokenizer chat template when available."""
    chat_template = getattr(tokenizer_obj, "chat_template", None)
    if chat_template:
        return tokenizer_obj.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    # Fallback if a chat template is unavailable.
    chunks = []
    for message in messages:
        role = message["role"].upper()
        chunks.append(f"{role}:\n{message['content']}")
    return "\n\n".join(chunks) + "\n\nASSISTANT:\n"


def prepare_example(example: dict[str, Any], noise_prob: float) -> dict[str, Any]:
    """Prepare cleaned/noisy transcript once and keep schema compact."""
    dialogue = clean_dialogue(example["dialogue"])
    if noise_prob > 0:
        dialogue = simulate_transcription_noise(dialogue, drop_prob=noise_prob)
    example["input_dialogue"] = dialogue
    return example


train_ds = train_ds.map(partial(prepare_example, noise_prob=CFG.train_noise_prob))
valid_ds = valid_ds.map(partial(prepare_example, noise_prob=CFG.valid_noise_prob))

print({"input_dialogue": train_ds[0]["input_dialogue"], "note": train_ds[0]["note"][:200] + "..."})

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

{'input_dialogue': "so sophia i see that you you hurt your knee tell me about what happened yeah i was jumping my kid's trampoline and i could just slipped out from under me my gosh one of those big trampolines in your back yard yeah a pretty big one okay which knee was it my right knee right knee okay and when did this happen about four days ago great the weather was perfect this weekend so i'm glad you at least got outside sorry to hear you got hurt okay so your right knee did you did you feel it pop or or snap or anything when you hurt it yeah i felt a little pop and then it swelled up really big okay you try anything for the pain i took some ibuprofen and i put some ice on it okay did that help a little bit but it's still really hard to get around alright and have you have you been able to stand on it or does that hurt too much it hurts quite a bit to stand but i am able to put weight on it okay alright and what part of the knee is it inside outside middle kind of that inside part 

In [ ]:
# Tokenization/collator helpers
tokenizer = AutoTokenizer.from_pretrained(CFG.model_id)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer has chat template: {bool(getattr(tokenizer, 'chat_template', None))}")

def collate_fn(examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for example in examples:
        full_messages = build_messages(
            dialogue=example["input_dialogue"],
            note=example["note"]
        )

        # Tokenize the COMPLETE conversation in one shot and get the mask back
        tokenized = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            return_assistant_tokens_mask=True,   # ← key fix
            return_dict=True,
            truncation=True,
            max_length=CFG.max_seq_length,
            add_special_tokens=False,
        )

        input_ids        = tokenized["input_ids"]
        attention_mask   = tokenized["attention_mask"]
        assistant_mask   = tokenized["assistant_tokens_mask"]  # 1 = assistant token

        labels = [
            token_id if is_assistant else -100
            for token_id, is_assistant in zip(input_ids, assistant_mask)
        ]

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)
        all_labels.append(labels)

    # Pad to the longest sequence in the batch
    max_len = max(len(ids) for ids in all_input_ids)

    def pad(seq, pad_val, max_len):
        return seq + [pad_val] * (max_len - len(seq))

    batch_input_ids      = torch.tensor([pad(x, tokenizer.pad_token_id, max_len) for x in all_input_ids])
    batch_attention_mask = torch.tensor([pad(x, 0, max_len) for x in all_attention_masks])
    batch_labels         = torch.tensor([pad(x, -100, max_len) for x in all_labels])

    return {
        "input_ids":      batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels":         batch_labels,
    }


def preprocess_logits_for_metrics(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Reduce logits to argmax token IDs before metric computation."""
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.argmax(logits, dim=-1)


def compute_token_metrics(eval_preds: tuple[np.ndarray, np.ndarray]) -> dict[str, float]:
    """Compute next-token accuracy over non-masked label positions."""
    pred_ids, label_ids = eval_preds

    pred_ids = np.asarray(pred_ids)
    label_ids = np.asarray(label_ids)

    shifted_preds = pred_ids[:, :-1]
    shifted_labels = label_ids[:, 1:]

    valid_mask = shifted_labels != -100
    valid_count = int(valid_mask.sum())
    if valid_count == 0:
        return {"token_accuracy": 0.0}

    correct = (shifted_preds == shifted_labels) & valid_mask
    token_accuracy = float(correct.sum() / valid_count)
    return {"token_accuracy": token_accuracy}


class EvalPerplexityCallback(TrainerCallback):
    """Adds eval_perplexity into eval metrics based on eval_loss."""

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        """Attach eval perplexity derived from eval loss."""
        if metrics is not None and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            if eval_loss is not None and eval_loss < 20:
                metrics["eval_perplexity"] = float(math.exp(eval_loss))


def load_causal_lm(
    model_id: str,
    tokenizer_obj: AutoTokenizer,
    is_h100: bool,
    for_training: bool = False,
) -> AutoModelForCausalLM:
    """Load model with a single hardware-aware path and aligned tokenizer ids."""
    if is_h100:
        loaded_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
            device_map={"": 0},
        )
    else:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        loaded_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quantization_config,
            dtype=torch.float16,
            device_map="auto",
        )

    loaded_model.config.pad_token_id = tokenizer_obj.pad_token_id
    loaded_model.config.eos_token_id = tokenizer_obj.eos_token_id
    loaded_model.config.bos_token_id = tokenizer_obj.bos_token_id
    if hasattr(loaded_model, "generation_config"):
        loaded_model.generation_config.pad_token_id = tokenizer_obj.pad_token_id
        loaded_model.generation_config.eos_token_id = tokenizer_obj.eos_token_id
        loaded_model.generation_config.bos_token_id = tokenizer_obj.bos_token_id

    if for_training:
        loaded_model.gradient_checkpointing_enable()
        loaded_model.config.use_cache = False

    return loaded_model

Tokenizer has chat template: True


In [ ]:
collate_fn(train_ds)

<bos><start_of_turn>user
You are an ambient clinical documentation assistant. Given a verbatim
transcript of a doctor-patient encounter, generate a structured
SOAP-style outpatient progress note.

### Guidelines
- Use clear ALL-CAPS section headings appropriate to the content
  (e.g., CHIEF COMPLAINT, HISTORY OF PRESENT ILLNESS, MEDICAL HISTORY,
  REVIEW OF SYSTEMS, VITALS, PHYSICAL EXAM, RESULTS, ASSESSMENT AND PLAN,
  INSTRUCTIONS).
- Write in a professional clinical style using third-person narrative prose.
- Include only sections for which information is present in the dialogue.
- Do not fabricate any information not explicitly stated in the dialogue.
- Preserve all medication names, doses, and frequencies exactly as spoken.
- The input transcript contains no speaker labels — infer clinical content
  from context.

<dialogue>
so sophia i see that you you hurt your knee tell me about what happened yeah i was jumping my kid's trampoline and i could just slipped out from under me my g

{'input_ids': tensor([[     1,      1,      1,  ...,  14210, 236813,      1],
        [     1,      1,      1,  ...,  14210, 236813,      1],
        [     1,      1,      1,  ...,  14210, 236813,      1],
        ...,
        [     1,      1,      1,  ...,  14210, 236813,      1],
        [     1,      1,      1,  ...,  14210, 236813,      1],
        [     1,      1,      1,  ...,  14210, 236813,      1]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'labels': tensor([[-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],

In [ ]:
# Model load (H100-optimized path with non-H100 fallback)
model = load_causal_lm(CFG.model_id, tokenizer, IS_H100, for_training=True)

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [ ]:
# LoRA config + attach adapters
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],# "all-linear",
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 3,223,552 || all params: 4,303,303,024 || trainable%: 0.0749


In [ ]:
# Trainer args + callbacks
eval_subset_size = min(CFG.eval_subset_size, len(valid_ds))
eval_ds_for_training = valid_ds.shuffle(seed=CFG.seed).select(range(eval_subset_size))

training_args = SFTConfig(
    output_dir=CFG.output_dir,
    hub_model_id=CFG.hub_model_id,
    num_train_epochs=CFG.num_train_epochs,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused" if IS_H100 else "paged_adamw_8bit",
    learning_rate=CFG.learning_rate,
    warmup_steps=CFG.warmup_steps,
    max_grad_norm=CFG.max_grad_norm,
    lr_scheduler_type="linear",
    fp16=False,
    bf16=True, # For some reason this is working for both old and new GPUs
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=CFG.eval_steps,
    save_steps=CFG.save_steps,
    logging_steps=CFG.logging_steps,
    logging_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=CFG.push_to_hub,
    report_to=CFG.report_to,
    seed=CFG.seed,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
    dataset_text_field=None,
    dataloader_num_workers=CFG.num_dataloader_workers,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=CFG.num_dataloader_workers > 0,
    torch_compile=IS_H100,
    torch_compile_backend="inductor" if IS_H100 else None,
    tf32=IS_H100,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds_for_training,
    processing_class=tokenizer,
    data_collator=collate_fn,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=CFG.early_stopping_patience),
        EvalPerplexityCallback(),
    ],
    compute_metrics=compute_token_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir {CFG.output_dir}

In [ ]:
# Train + best-checkpoint restore
train_result = trainer.train()

print(train_result)
print(trainer.state.best_model_checkpoint)

Step,Training Loss,Validation Loss,Token Accuracy,Perplexity
5,8.539219,6.852134,0.147636,945.897515
10,6.484725,5.671148,0.171051,290.367772
15,5.500480,4.808366,0.221378,122.531274
20,4.554242,4.226552,0.267599,68.480704
25,4.082846,3.720986,0.302874,41.305106
30,3.498064,3.087076,0.358370,21.912905
35,2.819098,2.318157,0.462673,10.156937
40,2.071250,1.678331,0.576099,5.356611
45,1.506286,1.205192,0.670823,3.337399
50,1.061092,0.821555,0.767979,2.274034


TrainOutput(global_step=108, training_loss=1.9391268470359069, metrics={'train_runtime': 1480.3589, 'train_samples_per_second': 0.284, 'train_steps_per_second': 0.073, 'total_flos': 1.838883187730304e+16, 'train_loss': 1.9391268470359069})
outputs/medgemma-aci-lora/checkpoint-105


In [ ]:
trainer.save_model()

In [ ]:
# Push best adapters to HF
if CFG.push_to_hub:
    model = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(CFG.model_id),
        trainer.state.best_model_checkpoint
    )

    model.push_to_hub(CFG.hub_model_id, token=hf_token)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  11%|#1        | 1.48MB / 12.9MB            

In [ ]:
# Evaluation pipeline + final metrics report
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def generate_predictions(eval_dataset, inference_model, tokenizer_obj, max_new_tokens: int) -> list[str]:
    """Generate note predictions for each evaluation sample."""
    predictions = []
    for i, example in enumerate(eval_dataset, 1):
        print(f"Testing sample {i} out of {len(eval_dataset)}")
        prompt_messages = extract_prompt_messages(build_messages(dialogue=example["input_dialogue"]))
        prompt_text = build_prompt_text(tokenizer_obj, prompt_messages)
        print(prompt_text)
        inputs = tokenizer_obj(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=CFG.max_seq_length,
            add_special_tokens=False,
        ).to(inference_model.device)

        with torch.no_grad():
            output_ids = inference_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=CFG.gen_repetition_penalty,
                # no_repeat_ngram_size=CFG.gen_no_repeat_ngram_size,
                pad_token_id=tokenizer_obj.eos_token_id,
                eos_token_id=tokenizer_obj.eos_token_id,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        generated_text = tokenizer_obj.decode(generated_ids, skip_special_tokens=True).strip()
        if CFG.verbose_eval and i <= CFG.verbose_eval_samples:
            # print(f"Expected:\n{example['note']}")
            print(generated_text)
        predictions.append(generated_text)
    return predictions


def compute_cross_entropy_and_perplexity(eval_dataset, eval_model, eval_tokenizer) -> tuple[float, float]:
    """Compute reference-conditioned CE/PPL with prompt tokens masked out."""
    eval_model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for example in eval_dataset:
            prompt_messages = extract_prompt_messages(build_messages(dialogue=example["input_dialogue"]))
            reference_text = wrap_note(example["note"])
            prompt_text = build_prompt_text(eval_tokenizer, prompt_messages)
            full_text = f"{prompt_text}{reference_text}{eval_tokenizer.eos_token}"

            inputs = eval_tokenizer(
                full_text,
                return_tensors="pt",
                truncation=True,
                max_length=CFG.max_seq_length,
                add_special_tokens=False,
            ).to(eval_model.device)

            prompt_ids = eval_tokenizer(
                prompt_text,
                truncation=True,
                max_length=CFG.max_seq_length,
                add_special_tokens=False,
            )["input_ids"]

            labels = inputs["input_ids"].clone()
            prompt_len = min(len(prompt_ids), labels.shape[1])
            labels[:, :prompt_len] = -100

            outputs = eval_model(**inputs, labels=labels)
            loss = outputs.loss
            token_count = int((labels != -100).sum().item())
            total_loss += float(loss.item() * token_count)
            total_tokens += token_count

    avg_ce = total_loss / max(total_tokens, 1)
    ppl = float(math.exp(avg_ce)) if avg_ce < 20 else float("inf")
    return avg_ce, ppl


def strip_note_wrapper(text: str) -> str:
    """Remove <note> wrapper tags from generated text for text metrics."""
    text = text.strip()
    text = re.sub(r"^\s*<note>\s*", "", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\s*</note>\s*$", "", text, flags=re.IGNORECASE | re.DOTALL)
    return text.strip()


def compute_text_metrics(eval_dataset, predictions: list[str], eval_model, eval_tokenizer) -> dict[str, float]:
    """Compute text overlap + semantic metrics and CE/PPL."""
    predictions = [strip_note_wrapper(p) for p in predictions]
    references = eval_dataset["note"]
    rouge = rouge_metric.compute(predictions=predictions, references=references)
    bleu = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    bertscore = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
    ce, ppl = compute_cross_entropy_and_perplexity(eval_dataset, eval_model, eval_tokenizer)

    return {
        "rouge1": float(rouge["rouge1"]),
        "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]),
        "bleu": float(bleu["bleu"]),
        "bertscore_f1": float(np.mean(bertscore["f1"])),
        "cross_entropy": ce,
        "perplexity": ppl,
    }


test_ds = concatenate_datasets([dataset["test1"], dataset["test2"], dataset["test3"]])
test_ds = test_ds.map(partial(prepare_example, noise_prob=CFG.test_noise_prob))

if CFG.test_subset_size > 0:
    test_ds = test_ds.select(range(min(CFG.test_subset_size, len(test_ds))))

def load_base_model_for_eval():
    """Load a fresh base model configured for evaluation-only comparisons."""
    return load_causal_lm(CFG.model_id, tokenizer, IS_H100, for_training=False)


print("\nEvaluating fine-tuned model on test prompts...")
finetuned_model = trainer.model
finetuned_model.config.use_cache = True  # ← restore before generation
finetuned_model.eval()
finetuned_predictions = generate_predictions(test_ds, finetuned_model, tokenizer, CFG.max_new_tokens)
finetuned_metrics = compute_text_metrics(test_ds, finetuned_predictions, finetuned_model, tokenizer)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nEvaluating base model on the same test prompts...")
base_model = load_base_model_for_eval()
base_model.eval()
base_predictions = generate_predictions(test_ds, base_model, tokenizer, CFG.max_new_tokens)
base_metrics = compute_text_metrics(test_ds, base_predictions, base_model, tokenizer)

print("\n===== Final Evaluation Metrics Comparison (Same Inputs) =====")
print("metric | finetuned | base | delta(ft-base)")
for metric_name in ["rouge1", "rouge2", "rougeL", "bleu", "bertscore_f1", "cross_entropy", "perplexity"]:
    ft_value = finetuned_metrics[metric_name]
    base_value = base_metrics[metric_name]
    delta = ft_value - base_value
    print(f"{metric_name} | {ft_value:.4f} | {base_value:.4f} | {delta:+.4f}")